In [ ]:

from pathlib import Path
from collections import Counter

# Your dataset location, based on your screenshots
DATA_ROOT = Path.home() / "Desktop" / "all spectogram" / "all 240 each"

print("Dataset folder exists:", DATA_ROOT.exists())
print("Dataset path:", DATA_ROOT)

all_images = []

for modality in ["pal", "per"]:
    folder = DATA_ROOT / modality

    if not folder.exists():
        print(f"\nFolder not found: {folder}")
        continue

    subject_folders = sorted(
        p for p in folder.iterdir() if p.is_dir()
    )

    counts = {
        p.name: len(list(p.glob("*.png")))
        for p in subject_folders
    }

    all_images.extend(folder.rglob("*.png"))

    print(f"\nModality: {modality}")
    print("Subject folders:", len(subject_folders))
    print("Total PNG images:", sum(counts.values()))
    print("Images per subject:", Counter(counts.values()))

print("\nTotal PNG images:", len(all_images))
print("Expected:", 18240)
print("Dataset count matches:", len(all_images) == 18240)


In [ ]:

import hashlib
from collections import defaultdict

hash_groups = defaultdict(list)

for path in all_images:
    with open(path, "rb") as f:
        file_hash = hashlib.sha256(f.read()).hexdigest()
    hash_groups[file_hash].append(path)

duplicate_groups = {
    h: paths for h, paths in hash_groups.items()
    if len(paths) > 1
}

duplicate_extra_files = sum(
    len(paths) - 1 for paths in duplicate_groups.values()
)

cross_class_groups = [
    paths for paths in duplicate_groups.values()
    if len({p.relative_to(DATA_ROOT).parts[0] for p in paths}) > 1
]

print("Total images checked:", len(all_images))
print("Unique file hashes:", len(hash_groups))
print("Exact duplicate groups:", len(duplicate_groups))
print("Extra duplicate files:", duplicate_extra_files)
print("Duplicate groups across pal/per:", len(cross_class_groups))

for i, paths in enumerate(list(duplicate_groups.values())[:5], 1):
    print(f"\nDuplicate group {i}:")
    for p in paths[:5]:
        print(" ", p.relative_to(DATA_ROOT))


In [ ]:

from collections import Counter

duplicate_subjects = Counter()
duplicate_modalities = Counter()
duplicate_across_subjects = 0

for paths in duplicate_groups.values():
    subjects = set()
    modalities = set()

    for p in paths:
        relative = p.relative_to(DATA_ROOT)
        modality = relative.parts[0]
        subject = relative.parts[1]

        modalities.add(modality)
        subjects.add(subject)

        duplicate_subjects[(modality, subject)] += 1

    if len(subjects) > 1:
        duplicate_across_subjects += 1

    for modality in modalities:
        duplicate_modalities[modality] += 1

print("Duplicate groups by modality:")
print(duplicate_modalities)

print("\nDuplicate groups by subject:")
for key, count in duplicate_subjects.most_common():
    print(key, ":", count)

print("\nDuplicate groups involving different subjects:",
      duplicate_across_subjects)


In [ ]:

from pathlib import Path

# Search only the current Jupyter project folder
# and its immediate subfolders.
project_folder = Path.cwd()

patterns = [
    "*manifest*.csv",
    "*split*.csv",
    "*subject*.json",
    "*split*.json",
    "*subject*.txt",
    "*split*.npy",
    "*split*.pkl"
]

found = set()

# Current folder
for pattern in patterns:
    for path in project_folder.glob(pattern):
        if path.is_file():
            found.add(path)

# Immediate subfolders only
for folder in project_folder.iterdir():
    if folder.is_dir():
        try:
            for pattern in patterns:
                for path in folder.glob(pattern):
                    if path.is_file():
                        found.add(path)
        except (OSError, PermissionError):
            continue

print("Current Jupyter folder:", project_folder)
print("\nPossible subject-split files:")

for path in sorted(found):
    print(path)

print("\nTotal candidate files:", len(found))


In [ ]:

from pathlib import Path

subject = "subject_31"

# Show the current notebook location
print("Current folder:", Path.cwd())

# Search likely experiment folders for subject split files
folders = [
    Path.home() / "Downloads",
    Path.home() / "Documents",
    Path.home() / "Desktop" / "all spectogram"
]

keywords = ("split", "manifest", "subject", "train", "test", "val")

for folder in folders:
    print(f"\nChecking: {folder}")

    if not folder.exists():
        continue

    try:
        for item in folder.iterdir():
            if any(word in item.name.lower() for word in keywords):
                print(" ", item.name)
    except OSError as e:
        print("  Cannot access:", e)

print("\nSubject to investigate:", subject)


In [ ]:

from PIL import Image
from collections import defaultdict
import hashlib
import numpy as np

pixel_groups = defaultdict(list)
failed = []

for path in all_images:
    try:
        with Image.open(path) as img:
            arr = np.asarray(img.convert("RGB"))

        h = hashlib.sha256()
        h.update(str(arr.shape).encode("utf-8"))
        h.update(arr.tobytes())
        pixel_groups[h.hexdigest()].append(path)

    except Exception as e:
        failed.append((str(path), str(e)))

pixel_duplicate_groups = {
    h: paths for h, paths in pixel_groups.items()
    if len(paths) > 1
}

pixel_extra_duplicates = sum(
    len(paths) - 1
    for paths in pixel_duplicate_groups.values()
)

print("Images checked:", len(all_images))
print("Failed images:", len(failed))
print("Unique pixel hashes:", len(pixel_groups))
print("Pixel-identical duplicate groups:",
      len(pixel_duplicate_groups))
print("Extra pixel-identical copies:",
      pixel_extra_duplicates)

subjects = defaultdict(int)

for paths in pixel_duplicate_groups.values():
    for path in paths:
        relative = path.relative_to(DATA_ROOT)
        subjects[(relative.parts[0], relative.parts[1])] += 1

print("\nDuplicate images involved by subject:")
for key, count in sorted(subjects.items()):
    print(key, count)

if failed:
    print("\nFirst five unreadable files:")
    print(failed[:5])


In [ ]:

# Install the imagehash library if necessary
import sys
import subprocess

try:
    import imagehash
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "ImageHash"
    ])
    import imagehash

from PIL import Image
from collections import defaultdict, Counter
from itertools import combinations

# Calculate perceptual hashes for all images
phash_groups = defaultdict(list)

for path in all_images:
    with Image.open(path) as img:
        h = str(imagehash.phash(img.convert("L")))
    phash_groups[h].append(path)

print("Total images:", len(all_images))
print("Unique perceptual hashes:", len(phash_groups))

same_phash_groups = [
    paths for paths in phash_groups.values()
    if len(paths) > 1
]

print("Groups sharing identical pHash:",
      len(same_phash_groups))

print("Additional images sharing pHash:",
      sum(len(paths)-1 for paths in same_phash_groups))

cross_subject = 0
cross_class = 0

for paths in same_phash_groups:
    subjects = {
        p.relative_to(DATA_ROOT).parts[1]
        for p in paths
    }
    classes = {
        p.relative_to(DATA_ROOT).parts[0]
        for p in paths
    }

    if len(subjects) > 1:
        cross_subject += 1

    if len(classes) > 1:
        cross_class += 1

print("Same-pHash groups across subjects:", cross_subject)
print("Same-pHash groups across classes:", cross_class)

print("\nFirst 5 matching groups:")
for paths in same_phash_groups[:5]:
    for p in paths[:3]:
        print(" ", p.relative_to(DATA_ROOT))
    print()


In [ ]:

import matplotlib.pyplot as plt
from PIL import Image

# Find pHash groups that span different subjects or classes
suspicious_groups = []

for paths in same_phash_groups:
    classes = {
        p.relative_to(DATA_ROOT).parts[0]
        for p in paths
    }
    subjects = {
        p.relative_to(DATA_ROOT).parts[1]
        for p in paths
    }

    if len(classes) > 1 or len(subjects) > 1:
        suspicious_groups.append(paths)

# Prioritize groups spanning percussion and palpation
suspicious_groups.sort(
    key=lambda paths: len({
        p.relative_to(DATA_ROOT).parts[0]
        for p in paths
    }) > 1,
    reverse=True
)

print("Suspicious groups:", len(suspicious_groups))

# Display first 3 groups, up to 3 images per group
for group_index, paths in enumerate(suspicious_groups[:3], 1):
    shown = paths[:3]

    fig, axes = plt.subplots(
        1, len(shown),
        figsize=(5 * len(shown), 4)
    )

    if len(shown) == 1:
        axes = [axes]

    for ax, path in zip(axes, shown):
        with Image.open(path) as img:
            ax.imshow(img.convert("L"), cmap="gray")

        ax.set_title(
            str(path.relative_to(DATA_ROOT)),
            fontsize=8
        )
        ax.axis("off")

    fig.suptitle(f"Matching pHash group {group_index}")
    plt.tight_layout()
    plt.show()


In [ ]:

import numpy as np
from PIL import Image
from collections import defaultdict

results = defaultdict(list)

for path in all_images:
    with Image.open(path) as img:
        gray = np.asarray(
            img.convert("L").resize((224, 224)),
            dtype=np.float32
        ) / 255.0

    modality = path.relative_to(DATA_ROOT).parts[0]

    results[modality].append({
        "path": str(path),
        "mean": float(gray.mean()),
        "std": float(gray.std()),
        "dark_fraction": float((gray < 0.05).mean())
    })

print("IMAGE BRIGHTNESS AND CONTRAST ANALYSIS")

for modality in ["pal", "per"]:
    records = results[modality]

    means = np.array([r["mean"] for r in records])
    stds = np.array([r["std"] for r in records])
    darks = np.array([r["dark_fraction"] for r in records])

    print(f"\nClass: {modality}")
    print("Images:", len(records))
    print("Average brightness:", round(float(means.mean()), 5))
    print("Average contrast (std):", round(float(stds.mean()), 5))
    print("Median dark-pixel fraction:",
          round(float(np.median(darks)), 5))

    print("Images with std < 0.01:",
          int((stds < 0.01).sum()))

    print("Images with > 95% very dark pixels:",
          int((darks > 0.95).sum()))

    print("Images with > 99% very dark pixels:",
          int((darks > 0.99).sum()))


In [ ]:

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    roc_auc_score, confusion_matrix
)
from sklearn.model_selection import GroupShuffleSplit

X, y, groups = [], [], []

for modality, label in [("pal", 0), ("per", 1)]:
    for record in results[modality]:
        X.append([
            record["mean"],
            record["std"],
            record["dark_fraction"]
        ])
        y.append(label)

        path = Path(record["path"])
        groups.append(path.parent.name)

X = np.asarray(X)
y = np.asarray(y)
groups = np.asarray(groups)

# Exploratory subject-wise holdout, not the original paper split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=6,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups)
)

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=2000,
        random_state=42
    )
)

model.fit(X[train_idx], y[train_idx])

pred = model.predict(X[test_idx])
prob = model.predict_proba(X[test_idx])[:, 1]

print("IMAGE-STATISTICS-ONLY CLASSIFICATION")
print("Training subjects:", len(set(groups[train_idx])))
print("Test subjects:", len(set(groups[test_idx])))

print("Training images:", len(train_idx))
print("Test images:", len(test_idx))

print("Accuracy:",
      round(accuracy_score(y[test_idx], pred), 4))
print("Balanced accuracy:",
      round(balanced_accuracy_score(y[test_idx], pred), 4))
print("ROC-AUC:",
      round(roc_auc_score(y[test_idx], prob), 4))

print("Confusion matrix:")
print(confusion_matrix(y[test_idx], pred))


In [ ]:

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np

splitter = GroupShuffleSplit(
    n_splits=20,
    test_size=6,
    random_state=42
)

accuracies = []
aucs = []

for train_idx, test_idx in splitter.split(X, y, groups):
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=42)
    )

    model.fit(X[train_idx], y[train_idx])

    predictions = model.predict(X[test_idx])
    probabilities = model.predict_proba(X[test_idx])[:, 1]

    accuracies.append(
        accuracy_score(y[test_idx], predictions)
    )

    aucs.append(
        roc_auc_score(y[test_idx], probabilities)
    )

print("REPEATED SUBJECT-WISE IMAGE-STATISTICS CHECK")
print("Number of splits:", len(accuracies))
print("Mean accuracy:", round(np.mean(accuracies), 4))
print("Accuracy std:", round(np.std(accuracies, ddof=1), 4))
print("Minimum accuracy:", round(np.min(accuracies), 4))
print("Maximum accuracy:", round(np.max(accuracies), 4))
print("Mean ROC-AUC:", round(np.mean(aucs), 4))
print("ROC-AUC std:", round(np.std(aucs, ddof=1), 4))
